# TimesFM × UCI Online Retail：Jupyter 實作教材

這份 Notebook 搭配同目錄的完整程式包使用，目標是讓你逐格看懂並重跑：

1. 讀取已處理的每日銷售資料。
2. 使用 chronological holdout 保留最後 28 天。
3. 重跑 Seasonal Naive(7) 與 ETS。
4. 計算 MAE、RMSE、sMAPE、WAPE。
5. 讀取本次已驗證的 TimesFM 3.0 輸出，公平比較三個模型。
6. 選擇性下載資料與 TimesFM 權重，另存完整重跑結果。

> 本案例的 target 是每日正向商品銷售毛額（GBP），不是淨收入、需求量或行銷增量。TimesFM 3.0 權重採 Non-Commercial License；預設不下載、不執行大型模型。


## 0. 執行環境

請先下載並解壓完整程式包，從程式包根目錄啟動 Jupyter。Baseline 所需套件：

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements-baseline.txt
.venv/bin/python -m pip install jupyterlab  # 只有本機尚未安裝 Jupyter 時才需要
.venv/bin/jupyter lab
```

Notebook 不會讀取帳號、Cookie、Token、GA4 或 CRM 資料。


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing


def find_project_root(start: Path) -> Path:
    """從目前目錄往上找程式包根目錄，找不到就明確失敗。"""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data/processed/daily_gross_sales.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "找不到 data/processed/daily_gross_sales.csv；"
        "請先解壓完整程式包，並從程式包根目錄啟動 Jupyter。"
    )


ROOT = find_project_root(Path.cwd())
DATA_PATH = ROOT / "data/processed/daily_gross_sales.csv"
OUTPUT_PATH = ROOT / "outputs/forecast_holdout.csv"
METRICS_PATH = ROOT / "outputs/metrics.json"
print(f"專案根目錄：{ROOT}")


## 1. 讀取與檢查每日資料

輸入是程式包內已處理的公開 UCI Online Retail 日資料。這裡只保留聚合後欄位，不使用客戶層級資料。


In [ ]:
daily = pd.read_csv(DATA_PATH, parse_dates=["date"])
daily = daily.sort_values("date").reset_index(drop=True)

required = {"date", "gross_sales_gbp"}
missing = required.difference(daily.columns)
assert not missing, f"缺少欄位：{sorted(missing)}"
assert daily["date"].is_monotonic_increasing
assert daily["date"].is_unique
expected_dates = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
assert daily["date"].tolist() == expected_dates.tolist(), "日期序列不連續"
assert np.isfinite(daily["gross_sales_gbp"]).all()
assert (daily["gross_sales_gbp"] >= 0).all()

summary = pd.Series({
    "開始日期": daily["date"].min().date(),
    "結束日期": daily["date"].max().date(),
    "日數": len(daily),
    "零銷售日": int((daily["gross_sales_gbp"] == 0).sum()),
    "平均每日毛額（GBP）": daily["gross_sales_gbp"].mean(),
})
summary


## 2. Chronological holdout

最後 28 天只用於評估；所有模型只看 holdout 之前的資料。不能 random split，否則模型可能先看到未來模式。


In [ ]:
HORIZON = 28
SEASONAL_PERIOD = 7
assert len(daily) > HORIZON + SEASONAL_PERIOD

train = daily.iloc[:-HORIZON].copy()
test = daily.iloc[-HORIZON:].copy()
y_train = train["gross_sales_gbp"].to_numpy(dtype=float)
y_test = test["gross_sales_gbp"].to_numpy(dtype=float)

split_info = pd.Series({
    "train_start": train["date"].min().date(),
    "train_end": train["date"].max().date(),
    "holdout_start": test["date"].min().date(),
    "holdout_end": test["date"].max().date(),
    "horizon": HORIZON,
})
split_info


## 3. 定義兩個可重跑模型與四項指標

- Seasonal Naive(7)：把訓練資料最後 7 天重複四次。
- ETS：加法趨勢、阻尼趨勢、加法週期，週期長度 7。


In [ ]:
def seasonal_naive(train_values: np.ndarray, horizon: int, period: int = 7) -> np.ndarray:
    if len(train_values) < period:
        raise ValueError("訓練資料短於季節週期")
    return np.resize(train_values[-period:], horizon).astype(float)


def ets_forecast(train_values: np.ndarray, horizon: int, period: int = 7) -> np.ndarray:
    model = ExponentialSmoothing(
        train_values,
        trend="add",
        damped_trend=True,
        seasonal="add",
        seasonal_periods=period,
        initialization_method="estimated",
    )
    fitted = model.fit(optimized=True, use_brute=False)
    return np.asarray(fitted.forecast(horizon), dtype=float)


def metric_values(actual: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = predicted - actual
    denominator = np.abs(actual) + np.abs(predicted)
    smape_terms = np.divide(
        2.0 * np.abs(error),
        denominator,
        out=np.zeros_like(error),
        where=denominator != 0,
    )
    return {
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "sMAPE (%)": float(100.0 * np.mean(smape_terms)),
        "WAPE (%)": float(100.0 * np.abs(error).sum() / np.abs(actual).sum()),
    }


In [ ]:
naive_pred = seasonal_naive(y_train, HORIZON, SEASONAL_PERIOD)
ets_pred = ets_forecast(y_train, HORIZON, SEASONAL_PERIOD)

baseline_metrics = pd.DataFrame({
    "Seasonal Naive(7)": metric_values(y_test, naive_pred),
    "ETS": metric_values(y_test, ets_pred),
}).T

# 這些 assert 是教材的最小可執行檢查：避免資料或公式悄悄改變。
assert np.isclose(baseline_metrics.loc["Seasonal Naive(7)", "MAE"], 15184.707259285717)
assert np.isclose(baseline_metrics.loc["ETS", "MAE"], 17457.93741020681)
baseline_metrics.round(2)


## 4. 看預測曲線，而不只看一個分數

三個模型都會在最後一天的極端尖峰失手。圖形用來看週期與尖峰；冠軍仍須用相同 holdout 上的多項指標判定。


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test["date"], y_test, label="實際值", color="#111827", linewidth=2.4, marker="o", markersize=3)
ax.plot(test["date"], naive_pred, label="Seasonal Naive(7)", color="#d97706", linewidth=2)
ax.plot(test["date"], ets_pred, label="ETS", color="#0f766e", linewidth=2, linestyle="--")
ax.set_title("28 天 holdout：實際每日銷售額與 baseline 預測")
ax.set_ylabel("GBP／日")
ax.grid(alpha=0.25)
ax.legend()
fig.autofmt_xdate()
plt.show()


## 5. 尖峰敏感度檢查

移除最後一天只為理解誤差是否完全由單一尖峰主導；這不是重新挑選 holdout，也不能取代 rolling-origin validation。


In [ ]:
sensitivity = pd.DataFrame({
    "Seasonal Naive(7)": metric_values(y_test[:-1], naive_pred[:-1]),
    "ETS": metric_values(y_test[:-1], ets_pred[:-1]),
}).T
sensitivity.round(2)


## 6. 讀取已驗證的 TimesFM 3.0 輸出

下一格讀取程式包內的既有 TimesFM 輸出；它不會重新下載或執行模型。這個證據界線很重要：看到輸出不等於你剛完成一次新的 TimesFM 推論。


In [ ]:
artifact = pd.read_csv(OUTPUT_PATH, parse_dates=["date"])
saved = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
assert len(artifact) == HORIZON
assert artifact["date"].tolist() == test["date"].tolist()
assert np.allclose(artifact["actual_gross_sales_gbp"], y_test)
assert artifact["timesfm_3_point"].notna().all()

timesfm_pred = artifact["timesfm_3_point"].to_numpy(dtype=float)
all_metrics = pd.DataFrame({
    "Seasonal Naive(7)": metric_values(y_test, naive_pred),
    "ETS": metric_values(y_test, ets_pred),
    "TimesFM 3.0（既有輸出）": metric_values(y_test, timesfm_pred),
}).T

assert np.isclose(all_metrics.loc["TimesFM 3.0（既有輸出）", "MAE"], saved["models"]["timesfm_3"]["mae"])
all_metrics.round(2).sort_values("MAE")


### 正確判讀

- Seasonal Naive(7) 在 MAE、RMSE、sMAPE、WAPE 四項都最佳。
- TimesFM 在 MAE、WAPE 排第二；ETS 在 RMSE、sMAPE 略優於 TimesFM。
- 只有 28 個 holdout 點，不能外推成「哪個模型永遠比較好」。
- 下一步應做 rolling-origin validation，再加入 ARIMA／SARIMAX、梯度提升與未來已知 covariates。


## 7. 選用：完整重跑 TimesFM

下格預設為 `False`，避免無意下載大型 checkpoint。若要執行，先閱讀 Hugging Face 上的 TimesFM 3.0 Non-Commercial License，安裝 `requirements-timesfm.txt`，再將旗標改為 `True`。輸出另存到 `outputs/notebook_timesfm/`，不覆蓋教材附帶的已驗證輸出。


In [ ]:
RUN_FULL_TIMESFM = False

if RUN_FULL_TIMESFM:
    subprocess.run(
        [sys.executable, str(ROOT / "scripts/download_data.py")],
        cwd=ROOT,
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            str(ROOT / "scripts/run_forecast.py"),
            "--timesfm", "required",
            "--output-dir", "outputs/notebook_timesfm",
        ],
        cwd=ROOT,
        check=True,
    )
    print("完成：outputs/notebook_timesfm/")
else:
    print("未執行 TimesFM：將 RUN_FULL_TIMESFM 改為 True 才會下載資料與模型。")


## 8. 舉一反三

你可以沿用相同骨架替換模型，但請維持相同時間切分：

1. ARIMA／SARIMA：先看 ACF、PACF 與殘差，再用 rolling origin 比較。
2. SARIMAX：只加入預測當下已知的未來排程、假日或預算。
3. LightGBM／XGBoost：lag 與 rolling 特徵必須先 `shift(1)`，避免當日答案進入當日特徵。
4. GA4／CRM：先定義 target、資料延遲與可用時間，再做 point-in-time correct join。
5. 因果 uplift：需要 treatment 與反事實設計，不能從這份 forecast 直接推論。
